Import libraries and data 

In [ ]:
import pandas as pd
from pathlib import Path

pm1 = pd.read_csv(
    "../data/raw/ratnapark/pm1.csv"
)

pm10 = pd.read_csv(
    "../data/raw/ratnapark/pm10.csv"
)

pm25 = pd.read_csv(
    "../data/raw/ratnapark/pm25.csv"
)

print("PM1:", pm1.shape)
print("PM10:", pm10.shape)
print("PM2.5:", pm25.shape)

PM1: (19613, 2)
PM10: (19610, 2)
PM2.5: (19627, 2)


## convert datetime

In [ ]:
pm1["datetime"] = pd.to_datetime(
    pm1["datetime"],
    utc=True
)

pm10["datetime"] = pd.to_datetime(
    pm10["datetime"],
    utc=True
)

pm25["datetime"] = pd.to_datetime(
    pm25["datetime"],
    utc=True
)

Merge pollutants

In [ ]:
test_raw = (
    pm1
    .merge(
        pm10,
        on="datetime",
        how="outer"
    )
    .merge(
        pm25,
        on="datetime",
        how="outer"
    )
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Shape:", test_raw.shape)

display(test_raw.head())
display(test_raw.tail())

Shape: (19642, 4)


,datetime,pm1,pm10,pm25
0,2026-08-02 00:00:00+00:00,69.500000,84.500000,81.099998
1,2026-08-02 00:01:00+00:00,120.599998,134.000000,127.500000
2,2026-08-02 00:02:00+00:00,181.300003,195.899994,190.699997
3,2026-08-02 00:03:00+00:00,218.199997,231.500000,226.800003
4,2026-08-02 00:04:00+00:00,87.199997,100.800003,89.000000


,datetime,pm1,pm10,pm25
19637,2026-08-16 05:55:00+00:00,11.1,24.400000,14.9
19638,2026-08-16 05:56:00+00:00,8.8,27.900000,12.1
19639,2026-08-16 05:57:00+00:00,8.0,24.400000,10.2
19640,2026-08-16 05:58:00+00:00,7.3,16.799999,9.4
19641,2026-08-16 05:59:00+00:00,7.5,12.300000,8.4


Check missing values

In [ ]:
print("Missing values:")
display(test_raw.isna().sum())

print("\nDate range:")
print(test_raw["datetime"].min())
print(test_raw["datetime"].max())

Missing values:


datetime     0
pm1         29
pm10        32
pm25        15
dtype: int64


Date range:
2026-08-02 00:00:00+00:00
2026-08-16 05:59:00+00:00


COnvert minute data to daily 

In [ ]:
test_daily = (
    test_raw
    .set_index("datetime")
    .resample("D")
    .mean(numeric_only=True)
    .reset_index()
)

test_daily = test_daily.rename(
    columns={
        "datetime": "date",
        "pm25": "pm2_5"
    }
)

test_daily = (
    test_daily
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shape:", test_daily.shape)

display(test_daily)

,date,pm1,pm10,pm2_5
0,2026-08-02 00:00:00+00:00,55.195013,64.022251,58.833538
1,2026-08-03 00:00:00+00:00,56.616693,75.393556,62.634817
2,2026-08-04 00:00:00+00:00,51.808746,69.281502,58.748684
3,2026-08-05 00:00:00+00:00,35.848899,49.294396,39.631938
4,2026-08-06 00:00:00+00:00,36.590483,49.773565,41.120997
5,2026-08-07 00:00:00+00:00,17.447983,31.665229,20.730946
6,2026-08-08 00:00:00+00:00,10.267731,20.642748,13.310965
7,2026-08-09 00:00:00+00:00,8.037196,17.865232,10.405691
8,2026-08-10 00:00:00+00:00,6.991672,16.479667,9.005760
9,2026-08-11 00:00:00+00:00,11.383264,26.751389,15.107986


Check daily data

In [ ]:
print("Date range:")
print(test_daily["date"].min())
print(test_daily["date"].max())

print("\nMissing values:")
display(test_daily.isna().sum())

Shape: (15, 4)

Date range:
2026-08-02 00:00:00+00:00
2026-08-16 00:00:00+00:00

Missing values:


date     0
pm1      0
pm10     0
pm2_5    0
dtype: int64

Create ewma features

In [7]:
test_daily["pm2_5_ewma_3"] = (
    test_daily["pm2_5"]
    .shift(1)
    .ewm(
        span=3,
        adjust=False
    )
    .mean()
)

test_daily["pm2_5_ewma_7"] = (
    test_daily["pm2_5"]
    .shift(1)
    .ewm(
        span=7,
        adjust=False
    )
    .mean()
)

test_daily["pm10_ewma_3"] = (
    test_daily["pm10"]
    .shift(1)
    .ewm(
        span=3,
        adjust=False
    )
    .mean()
)

test_daily["pm10_ewma_7"] = (
    test_daily["pm10"]
    .shift(1)
    .ewm(
        span=7,
        adjust=False
    )
    .mean()
)

Define feature columns

In [8]:
pm25_features = [
    "pm2_5_ewma_3",
    "pm2_5_ewma_7",
    "pm10_ewma_3",
    "pm10_ewma_7",
]

print("PM2.5 features:")
for feature in pm25_features:
    print("-", feature)

PM2.5 features:
- pm2_5_ewma_3
- pm2_5_ewma_7
- pm10_ewma_3
- pm10_ewma_7


Select 15 days periods

In [9]:
test_period = test_daily[
    test_daily["date"].between(
        "2026-08-02",
        "2026-08-16"
    )
].copy()

test_period = (
    test_period
    .sort_values("date")
    .reset_index(drop=True)
)

print("Rows in test period:", len(test_period))

display(
    test_period[
        ["date", "pm2_5", "pm10"]
        + pm25_features
    ]
)

Rows in test period: 15


,date,pm2_5,pm10,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7
0,2026-08-02 00:00:00+00:00,58.833538,64.022251,NaN,NaN,NaN,NaN
1,2026-08-03 00:00:00+00:00,62.634817,75.393556,58.833538,58.833538,64.022251,64.022251
2,2026-08-04 00:00:00+00:00,58.748684,69.281502,60.734178,59.783858,69.707904,66.865077
3,2026-08-05 00:00:00+00:00,39.631938,49.294396,59.741431,59.525064,69.494703,67.469184
4,2026-08-06 00:00:00+00:00,41.120997,49.773565,49.686684,54.551783,59.394549,62.925487
5,2026-08-07 00:00:00+00:00,20.730946,31.665229,45.403841,51.194086,54.584057,59.637506
6,2026-08-08 00:00:00+00:00,13.310965,20.642748,33.067393,43.578301,43.124643,52.644437
7,2026-08-09 00:00:00+00:00,10.405691,17.865232,23.189179,36.011467,31.883696,44.644015
8,2026-08-10 00:00:00+00:00,9.005760,16.479667,16.797435,29.610023,24.874464,37.949319
9,2026-08-11 00:00:00+00:00,15.107986,26.751389,12.901597,24.458957,20.677066,32.581906


Remove unusable data 

In [10]:
test_pm25 = test_period.dropna(
    subset=pm25_features
).copy()

test_pm25 = (
    test_pm25
    .sort_values("date")
    .reset_index(drop=True)
)

print("Valid PM2.5 test rows:", len(test_pm25))

display(
    test_pm25[
        ["date", "pm2_5", "pm10"]
        + pm25_features
    ]
)

Valid PM2.5 test rows: 14


,date,pm2_5,pm10,pm2_5_ewma_3,pm2_5_ewma_7,pm10_ewma_3,pm10_ewma_7
0,2026-08-03 00:00:00+00:00,62.634817,75.393556,58.833538,58.833538,64.022251,64.022251
1,2026-08-04 00:00:00+00:00,58.748684,69.281502,60.734178,59.783858,69.707904,66.865077
2,2026-08-05 00:00:00+00:00,39.631938,49.294396,59.741431,59.525064,69.494703,67.469184
3,2026-08-06 00:00:00+00:00,41.120997,49.773565,49.686684,54.551783,59.394549,62.925487
4,2026-08-07 00:00:00+00:00,20.730946,31.665229,45.403841,51.194086,54.584057,59.637506
5,2026-08-08 00:00:00+00:00,13.310965,20.642748,33.067393,43.578301,43.124643,52.644437
6,2026-08-09 00:00:00+00:00,10.405691,17.865232,23.189179,36.011467,31.883696,44.644015
7,2026-08-10 00:00:00+00:00,9.005760,16.479667,16.797435,29.610023,24.874464,37.949319
8,2026-08-11 00:00:00+00:00,15.107986,26.751389,12.901597,24.458957,20.677066,32.581906
9,2026-08-12 00:00:00+00:00,14.218958,26.608750,14.004792,22.121214,23.714227,31.124277


In [ ]:
print("Missing values in required features:")
display(
    test_pm25[pm25_features].isna().sum()
)

Missing values in required features:


pm2_5_ewma_3    0
pm2_5_ewma_7    0
pm10_ewma_3     0
pm10_ewma_7     0
dtype: int64

Survived dates

In [12]:
print("Final PM2.5 test dates:")

for date in test_pm25["date"]:
    print(date.date())

Final PM2.5 test dates:
2026-08-03
2026-08-04
2026-08-05
2026-08-06
2026-08-07
2026-08-08
2026-08-09
2026-08-10
2026-08-11
2026-08-12
2026-08-13
2026-08-14
2026-08-15
2026-08-16


Save test dataset

In [13]:
output_path = Path(
    "../data/processed/ratnapark/"
    "ratnapark_pm25_test_15days.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

test_pm25[
    ["date", "pm2_5", "pm10"]
    + pm25_features
].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", test_pm25.shape)

Saved: ..\data\processed\ratnapark\ratnapark_pm25_test_15days.csv
Shape: (14, 8)
